# Module 1 Assignment: Foundations of Machine Learning
**Total: 100 marks | Questions: Q1–Q28**

This notebook implements all five assignment parts using Python, NumPy, pandas, Matplotlib, Seaborn and scikit-learn. K-Means in Part 5 is implemented from scratch; `sklearn.cluster.KMeans` is not used.

## How to run
1. Open this notebook in Google Colab or Jupyter.
2. Run **Runtime → Run all** (Colab) or **Kernel → Restart & Run All** (Jupyter).
3. `fetch_california_housing()` may download the California Housing dataset the first time it runs, so an internet connection may be required.
4. Random seeds are set where appropriate to make the results reproducible.

> **Assignment-specific note:** Part 1 explicitly asks for `MedHouseVal` as the input and `MedInc` as the target. The notebook follows that direction, even though the reverse is more commonly used in housing-price prediction.


## Setup: import libraries and load datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Consistent plot appearance
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Load the California Housing dataset
california = fetch_california_housing(as_frame=True)
california_df = california.frame.copy()

# Load the Iris dataset
iris = load_iris(as_frame=True)
iris_df = iris.frame.copy()

print("California Housing shape:", california_df.shape)
print("Iris shape:", iris_df.shape)
print("\nCalifornia Housing columns:")
print(california_df.columns.tolist())
print("\nIris columns:")
print(iris_df.columns.tolist())

# Part 1: Linear Regression (10 marks)

**Dataset:** California Housing  
**Goal:** Fit a simple linear regression using one input feature. The assignment specifies `MedHouseVal` as input and `MedInc` as target.


### Q1 (2 marks): Print the feature names of the dataframe

In [ ]:
# Q1: Print all feature/column names in the California Housing DataFrame.
print(california_df.columns.tolist())

### Q2 (2 marks): Set MedHouseVal as input and MedInc as target; print shapes

`X` contains the input feature and `y` contains the target. Double brackets keep `X` as a 2D DataFrame, as expected by scikit-learn.


In [ ]:
# Q2: Follow the exact input/target direction stated in the assignment.
X_lr = california_df[["MedHouseVal"]]
y_lr = california_df["MedInc"]

print("Input (X) shape:", X_lr.shape)
print("Target (y) shape:", y_lr.shape)

### Q3 (2 marks): Visualize the samples as a 2D scatter plot

In [ ]:
# Q3: Plot MedHouseVal on the x-axis and MedInc on the y-axis.
plt.figure(figsize=(8, 5))
plt.scatter(X_lr["MedHouseVal"], y_lr, alpha=0.25, s=12)
plt.xlabel("MedHouseVal (input)")
plt.ylabel("MedInc (target)")
plt.title("California Housing: MedInc vs MedHouseVal")
plt.tight_layout()
plt.show()

### Q4 (2 marks): Fit a linear regressor and print slope and intercept

A simple linear model has the form:

\[
\hat y = m x + b
\]

where `m` is the slope (`coef_`) and `b` is the intercept (`intercept_`).


In [ ]:
# Q4: Fit Linear Regression using the full dataset (no split is required in Part 1).
linear_model = LinearRegression()
linear_model.fit(X_lr, y_lr)

slope = float(linear_model.coef_[0])
intercept = float(linear_model.intercept_)

print("Slope:", slope)
print("Intercept:", intercept)

### Q5 (2 marks): Plot the fitted regression line over the scatter plot

In [ ]:
# Q5: Predict over a sorted range so the regression line is drawn smoothly.
x_line = np.linspace(X_lr["MedHouseVal"].min(), X_lr["MedHouseVal"].max(), 300)
y_line = linear_model.predict(pd.DataFrame({"MedHouseVal": x_line}))

plt.figure(figsize=(8, 5))
plt.scatter(X_lr["MedHouseVal"], y_lr, alpha=0.20, s=12, label="Samples")
plt.plot(x_line, y_line, color="red", linewidth=2.5, label="Regression line")
plt.xlabel("MedHouseVal (input)")
plt.ylabel("MedInc (target)")
plt.title("Linear Regression: MedInc vs MedHouseVal")
plt.legend()
plt.tight_layout()
plt.show()

# Part 2: Logistic Regression (10 marks)

**Dataset:** Iris  
**Goal:** Convert the three Iris labels to integers, remove `versicolor`, and perform binary classification between `setosa` and `virginica`.


### Q6 (2 marks): Print input and output features for five samples

In [ ]:
# Q6: Display five rows of input features and their original target labels.
iris_feature_columns = iris.feature_names
print("Five input samples:")
print(iris_df[iris_feature_columns].head(5))

print("\nCorresponding output labels:")
print(iris_df["target"].head(5).map(dict(enumerate(iris.target_names))))

### Q7 (2 marks): Visualize the features using a pairplot

The pairplot shows pairwise relationships among the four numeric Iris features. Hue represents the original three species.


In [ ]:
# Q7: Use readable species names for the pairplot legend.
iris_pairplot_df = iris_df[iris_feature_columns].copy()
iris_pairplot_df["species"] = iris_df["target"].map(dict(enumerate(iris.target_names)))

sns.pairplot(iris_pairplot_df, hue="species", diag_kind="hist")
plt.suptitle("Iris Feature Pairplot", y=1.02)
plt.show()

### Q8 (2 marks): Convert labels, remove the middle class, and relabel virginica

Required mapping before filtering: `setosa = 0`, `versicolor = 1`, `virginica = 2`. Then remove class 1 and change class 2 to class 1. The final binary labels are therefore `setosa = 0` and `virginica = 1`.


In [ ]:
# Q8: Apply the requested label conversion and filtering.
iris_binary_df = iris_df.copy()
iris_binary_df["target"] = iris_binary_df["target"].astype(int)  # 0, 1, 2

# Remove versicolor (class 1).
iris_binary_df = iris_binary_df[iris_binary_df["target"] != 1].copy()

# Change virginica from 2 to 1. Setosa remains 0.
iris_binary_df["target"] = iris_binary_df["target"].replace({2: 1})

print("Final binary class counts:")
print(iris_binary_df["target"].value_counts().sort_index())
print("Final labels:", sorted(iris_binary_df["target"].unique()))

### Q9 (2 marks): Split X and y into 80% training and 20% testing subsets

In [ ]:
# Q9: Use all four Iris features for logistic regression.
X_log = iris_binary_df[iris_feature_columns]
y_log = iris_binary_df["target"]

X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(
    X_log, y_log,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_log
)

print("X train:", X_log_train.shape)
print("X test:", X_log_test.shape)
print("y train:", y_log_train.shape)
print("y test:", y_log_test.shape)

### Q10 (2 marks): Train Logistic Regression, print weights and bias, and evaluate accuracy

For binary logistic regression, `coef_` contains the feature weights and `intercept_` contains the bias.


In [ ]:
# Q10: Train and evaluate the binary logistic regression model.
logistic_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logistic_model.fit(X_log_train, y_log_train)

y_log_pred = logistic_model.predict(X_log_test)

print("Weights (coefficients):")
print(logistic_model.coef_)
print("Bias (intercept):")
print(logistic_model.intercept_)
print("Test accuracy:", accuracy_score(y_log_test, y_log_pred))

# Part 3: Support Vector Classifier (20 marks)

**Dataset:** Iris, restricted to petal length and petal width and to two classes (`setosa` and `versicolor`).  
**Goal:** Compare linear SVM decision boundaries for `C=1` and `C=1000`.

`LinearSVC(loss="hinge")` implements a linear support vector classifier with hinge loss. Larger `C` penalizes training errors more strongly and generally allows less regularization.


### Q11 (2 marks): Remove virginica and print five samples

In [ ]:
# Q11: Keep only setosa (0) and versicolor (1), and select the two requested features.
svm_df = iris_df.loc[iris_df["target"] != 2,
                     ["petal length (cm)", "petal width (cm)", "target"]].copy()

print(svm_df.head(5))

### Q12 (2 marks): Plot the two input features with different class colors

In [ ]:
# Q12: Two-dimensional scatter plot of petal length and petal width.
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=svm_df,
    x="petal length (cm)", y="petal width (cm)",
    hue="target", palette="Set1", s=65
)
plt.title("Iris: Setosa vs Versicolor")
plt.tight_layout()
plt.show()

### Q13 (2 marks): Create an 80/20 train-test split

In [ ]:
# Q13: Split the two-feature dataset into training and testing subsets.
X_svm = svm_df[["petal length (cm)", "petal width (cm)"]]
y_svm = svm_df["target"]

X_svm_train, X_svm_test, y_svm_train, y_svm_test = train_test_split(
    X_svm, y_svm,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_svm
)

print("X train:", X_svm_train.shape)
print("X test:", X_svm_test.shape)
print("y train:", y_svm_train.shape)
print("y test:", y_svm_test.shape)

### Q14 (7 marks): Train Linear SVC with C=1, plot decision boundary and margins, and print accuracy

The decision boundary is the zero contour of the model's decision function. The two margin lines are the contours at `-1` and `+1`. The model is trained only on the training subset; the plot displays all samples.


In [ ]:
# Helper function used to plot the decision boundary and margins for a fitted linear SVC.
def plot_svm_boundary(model, X_all, y_all, title):
    x_min, x_max = X_all.iloc[:, 0].min() - 0.5, X_all.iloc[:, 0].max() + 0.5
    y_min, y_max = X_all.iloc[:, 1].min() - 0.5, X_all.iloc[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )
    grid = pd.DataFrame(
        np.c_[xx.ravel(), yy.ravel()],
        columns=X_all.columns
    )
    decision_values = model.decision_function(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, decision_values, levels=[-100, 0, 100],
                 alpha=0.12, colors=["#f4a6a6", "#9ecae1"])
    plt.contour(xx, yy, decision_values, levels=[-1, 0, 1],
                colors=["black", "black", "black"],
                linestyles=["--", "-", "--"],
                linewidths=[1.2, 2.0, 1.2])

    plt.scatter(
        X_all.iloc[:, 0], X_all.iloc[:, 1],
        c=y_all, cmap="coolwarm", edgecolors="k", s=55
    )
    plt.xlabel(X_all.columns[0])
    plt.ylabel(X_all.columns[1])
    plt.title(title)
    plt.tight_layout()
    plt.show()


# Q14: Linear SVC with hinge loss and C=1.
svm_c1 = LinearSVC(C=1, loss="hinge", dual=True,
                   max_iter=100000, random_state=RANDOM_STATE)
svm_c1.fit(X_svm_train, y_svm_train)

svm_c1_pred = svm_c1.predict(X_svm_test)
print("C=1 test accuracy:", accuracy_score(y_svm_test, svm_c1_pred))
print("Weights:", svm_c1.coef_)
print("Bias:", svm_c1.intercept_)

plot_svm_boundary(
    svm_c1, X_svm, y_svm,
    "Linear SVC (hinge loss), C=1: boundary and margins"
)

### Q15 (7 marks): Train Linear SVC with C=1000, plot decision boundary and margins, and print accuracy

This repeats Q14 with a much larger regularization parameter. Compare the boundary, margins and test accuracy with the `C=1` result.


In [ ]:
# Q15: Linear SVC with hinge loss and C=1000.
svm_c1000 = LinearSVC(C=1000, loss="hinge", dual=True,
                      max_iter=100000, random_state=RANDOM_STATE)
svm_c1000.fit(X_svm_train, y_svm_train)

svm_c1000_pred = svm_c1000.predict(X_svm_test)
print("C=1000 test accuracy:", accuracy_score(y_svm_test, svm_c1000_pred))
print("Weights:", svm_c1000.coef_)
print("Bias:", svm_c1000.intercept_)

plot_svm_boundary(
    svm_c1000, X_svm, y_svm,
    "Linear SVC (hinge loss), C=1000: boundary and margins"
)

# Part 4: Bagging (30 marks)

**Dataset:** California Housing  
**Goal:** Implement bootstrap aggregating manually using multiple `LinearRegression` base learners.

Bootstrap sampling selects training-row indices **with replacement**. Therefore, a bootstrap dataset has the same number of rows as the original training set, but some rows may appear multiple times while others may be absent. Ensemble predictions are the arithmetic mean of the individual models' predictions.


### Q16 (5 marks): Write a bootstrap-index method and split the data 80/20

The helper `get_bootstrap_indices` returns random row positions from the training set, sampled with replacement. These positions are then used to construct each bootstrap training sample.


In [ ]:
# Q16: Prepare the California Housing feature matrix and target.
X_bag = california_df.drop(columns=["MedHouseVal"])
y_bag = california_df["MedHouseVal"]

X_bag_train, X_bag_test, y_bag_train, y_bag_test = train_test_split(
    X_bag, y_bag,
    test_size=0.20,
    random_state=RANDOM_STATE
)

def get_bootstrap_indices(n_samples, random_state=None):
    """Return n_samples row indices sampled uniformly WITH replacement."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, size=n_samples, replace=True)

def make_bootstrap_sample(X_train, y_train, random_state=None):
    """Create and return a bootstrap-resampled X/y training pair."""
    indices = get_bootstrap_indices(len(X_train), random_state=random_state)
    X_bootstrap = X_train.iloc[indices].reset_index(drop=True)
    y_bootstrap = y_train.iloc[indices].reset_index(drop=True)
    return X_bootstrap, y_bootstrap, indices

print("X train:", X_bag_train.shape)
print("X test:", X_bag_test.shape)
print("y train:", y_bag_train.shape)
print("y test:", y_bag_test.shape)

# Demonstrate that a bootstrap sample has the same row count as the training set.
X_boot_demo, y_boot_demo, boot_indices_demo = make_bootstrap_sample(
    X_bag_train, y_bag_train, random_state=100
)
print("Bootstrap X shape:", X_boot_demo.shape)
print("Unique sampled rows:", len(np.unique(boot_indices_demo)))
print("Repeated/omitted rows are expected in bootstrap sampling.")

### Q17 (3 marks): Train one linear regression model on a bootstrap sample and print test MSE

In [ ]:
# Q17: Train the first base learner on a bootstrap variation of the training data.
bagging_models = []

X_boot_1, y_boot_1, _ = make_bootstrap_sample(
    X_bag_train, y_bag_train, random_state=101
)
model_1 = LinearRegression()
model_1.fit(X_boot_1, y_boot_1)
bagging_models.append(model_1)

pred_1_test = model_1.predict(X_bag_test)
mse_1 = mean_squared_error(y_bag_test, pred_1_test)
print("Test MSE using 1 bootstrap model:", mse_1)

### Q18 (4 marks): Train a second model and print MSE of the average of two predictions

In [ ]:
# Q18: Train a second model on a new bootstrap sample, then average both predictions.
X_boot_2, y_boot_2, _ = make_bootstrap_sample(
    X_bag_train, y_bag_train, random_state=102
)
model_2 = LinearRegression()
model_2.fit(X_boot_2, y_boot_2)
bagging_models.append(model_2)

pred_2_test = model_2.predict(X_bag_test)
avg_2_test = np.mean([model.predict(X_bag_test) for model in bagging_models], axis=0)
mse_2 = mean_squared_error(y_bag_test, avg_2_test)
print("Test MSE of the average of 2 models:", mse_2)

### Q19 (4 marks): Train a third model and print MSE of the average of three predictions

In [ ]:
# Q19: Train a third model using a fresh bootstrap sample.
X_boot_3, y_boot_3, _ = make_bootstrap_sample(
    X_bag_train, y_bag_train, random_state=103
)
model_3 = LinearRegression()
model_3.fit(X_boot_3, y_boot_3)
bagging_models.append(model_3)

avg_3_test = np.mean([model.predict(X_bag_test) for model in bagging_models], axis=0)
mse_3 = mean_squared_error(y_bag_test, avg_3_test)
print("Test MSE of the average of 3 models:", mse_3)

### Q20 (4 marks): Train a fourth model and print MSE of the average of four predictions

In [ ]:
# Q20: Train a fourth model using another fresh bootstrap sample.
X_boot_4, y_boot_4, _ = make_bootstrap_sample(
    X_bag_train, y_bag_train, random_state=104
)
model_4 = LinearRegression()
model_4.fit(X_boot_4, y_boot_4)
bagging_models.append(model_4)

avg_4_test = np.mean([model.predict(X_bag_test) for model in bagging_models], axis=0)
mse_4 = mean_squared_error(y_bag_test, avg_4_test)
print("Test MSE of the average of 4 models:", mse_4)

### Q21 (10 marks): Evaluate ensembles of 2, 4, 6, ..., 30 models and plot training/testing loss

For each ensemble size:
- Train that many base linear regression models, each on its own bootstrap sample.
- Average their predictions on the original training and test sets.
- Calculate training MSE and testing MSE for the averaged predictions.
- Plot both loss curves against ensemble size.

The training loss is calculated on the original `X_bag_train`, not on the resampled bootstrap rows.


In [ ]:
# Q21: Build ensembles with 2, 4, 6, ..., 30 bootstrap-trained models.
ensemble_sizes = list(range(2, 31, 2))
training_mse_values = []
testing_mse_values = []

# Use a separate deterministic seed sequence for this experiment.
for n_models in ensemble_sizes:
    current_models = []

    for model_number in range(n_models):
        seed = 1000 + n_models * 100 + model_number
        X_boot, y_boot, _ = make_bootstrap_sample(
            X_bag_train, y_bag_train, random_state=seed
        )

        base_model = LinearRegression()
        base_model.fit(X_boot, y_boot)
        current_models.append(base_model)

    # Average predictions across all base learners.
    train_predictions = np.mean(
        [model.predict(X_bag_train) for model in current_models], axis=0
    )
    test_predictions = np.mean(
        [model.predict(X_bag_test) for model in current_models], axis=0
    )

    train_mse = mean_squared_error(y_bag_train, train_predictions)
    test_mse = mean_squared_error(y_bag_test, test_predictions)

    training_mse_values.append(train_mse)
    testing_mse_values.append(test_mse)

    print(
        f"Models: {n_models:2d} | "
        f"Training MSE: {train_mse:.6f} | "
        f"Testing MSE: {test_mse:.6f}"
    )

# Plot training and testing loss for every ensemble size.
plt.figure(figsize=(9, 5))
plt.plot(ensemble_sizes, training_mse_values, marker="o", label="Training MSE")
plt.plot(ensemble_sizes, testing_mse_values, marker="s", label="Testing MSE")
plt.xlabel("Number of models in the ensemble")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Bagging: Training and Testing Loss vs Ensemble Size")
plt.xticks(ensemble_sizes)
plt.legend()
plt.tight_layout()
plt.show()

bagging_results = pd.DataFrame({
    "Number of Models": ensemble_sizes,
    "Training MSE": training_mse_values,
    "Testing MSE": testing_mse_values
})
display(bagging_results)

## Part 4: Short interpretation

Bagging trains base learners on different bootstrap samples and averages their predictions. Averaging can reduce variance, so test error may improve as the number of models increases. The curve need not decrease at every step; the observed values depend on the dataset, bootstrap samples and base learner. For linear regression, the models are relatively stable, so the improvement may be modest.


# Part 5: K-Means Clustering from Scratch (30 marks)

**Dataset:** Iris, using only `petal length (cm)` and `petal width (cm)`.  
**Important:** The clustering algorithm below is implemented manually. It does not call `sklearn.cluster.KMeans`.

K-Means alternates between:
1. **Assignment step:** assign each point to its nearest centroid using Euclidean distance.
2. **Update step:** replace each centroid with the mean of the points assigned to that cluster.
3. Repeat until the centroids stop changing (within a small numerical tolerance).


### Prepare the two-dimensional points dataset

The variable `points` is the global dataset used by the helper functions in Q22–Q27.


In [ ]:
# Prepare 2D Iris points: petal length and petal width.
points = iris_df[["petal length (cm)", "petal width (cm)"]].to_numpy(dtype=float)

print("Points shape:", points.shape)
print("First five points:")
print(points[:5])

### Q22 (3 marks): Find the index of the closest centroid

For a data point \(x\) and centroid \(c_j\), Euclidean distance is:

\[
d(x,c_j) = \sqrt{\sum_i (x_i-c_{j,i})^2}
\]

The function returns the index of the centroid with the smallest distance.


In [ ]:
# Q22: Return the index of the closest centroid to one data point.
def closest_centroid_index(point, centroids):
    point = np.asarray(point, dtype=float)
    centroids = np.asarray(centroids, dtype=float)

    distances = np.linalg.norm(centroids - point, axis=1)
    return int(np.argmin(distances))

# Small example to verify the function.
example_centroids = np.array([[1.0, 1.0], [4.0, 4.0], [7.0, 7.0]])
example_point = np.array([3.5, 3.8])
print("Closest centroid index:", closest_centroid_index(example_point, example_centroids))

### Q23 (3 marks): Calculate updated centroids from a cluster dictionary

The input dictionary maps cluster IDs to lists of 2D points. Each new centroid is the coordinate-wise mean of the points in its cluster. Empty clusters are represented by `NaN`; the main K-Means loop will retain the previous centroid for an empty cluster.


In [ ]:
# Q23: Calculate a centroid for each cluster in cluster_dict.
def calculate_new_centroids(cluster_dict):
    cluster_ids = sorted(cluster_dict.keys())
    new_centroids = []

    for cluster_id in cluster_ids:
        cluster_points = np.asarray(cluster_dict[cluster_id], dtype=float)

        if len(cluster_points) == 0:
            # There is no mean for an empty cluster; the main loop handles it.
            new_centroids.append([np.nan, np.nan])
        else:
            new_centroids.append(cluster_points.mean(axis=0))

    return np.asarray(new_centroids, dtype=float)

# Example verification.
example_clusters = {
    0: [[1.0, 2.0], [3.0, 4.0]],
    1: [[5.0, 6.0], [7.0, 8.0]]
}
print("Updated example centroids:")
print(calculate_new_centroids(example_clusters))

### Q24 (3 marks): Select p initial centroids using a random seed

The function sets the NumPy random seed and selects `p` distinct rows from the globally defined `points` dataset. Sampling without replacement avoids choosing the same data row more than once for initialization.


In [ ]:
# Q24: Randomly select p initial centroids from the global points dataset.
def initialize_centroids(random_seed, p):
    if p < 1 or p > len(points):
        raise ValueError("p must be between 1 and the number of available points.")

    np.random.seed(random_seed)
    selected_indices = np.random.choice(len(points), size=p, replace=False)
    return points[selected_indices].copy()

initial_centroids_example = initialize_centroids(42, 3)
print("Initial centroids (p=3):")
print(initial_centroids_example)

### Q25 (3 marks): Calculate Sum of Squared Errors (SSE)

SSE is the sum of squared Euclidean distances from every point to the centroid of its assigned cluster:

\[
SSE = \sum_{j=1}^{k}\sum_{x\in C_j}\|x-\mu_j\|^2
\]


In [ ]:
# Q25: Calculate the total within-cluster sum of squared errors.
def calculate_sse(centroids, cluster_dict, k):
    centroids = np.asarray(centroids, dtype=float)
    total_sse = 0.0

    for cluster_id in range(k):
        cluster_points = np.asarray(cluster_dict[cluster_id], dtype=float)

        if len(cluster_points) == 0:
            continue

        differences = cluster_points - centroids[cluster_id]
        squared_distances = np.sum(differences ** 2, axis=1)
        total_sse += float(np.sum(squared_distances))

    return total_sse

# Example verification using the example clusters and their centroids.
example_centroids_updated = calculate_new_centroids(example_clusters)
example_sse = calculate_sse(example_centroids_updated, example_clusters, 2)
print("Example SSE:", example_sse)

### Q26 (6 marks): Implement the complete K-Means loop from scratch

The function accepts the number of clusters `p`, the points to cluster, and the initial centroid array. It returns:
- the final centroid array; and
- a dictionary where each cluster ID maps to its assigned points.

The loop stops when centroids no longer change within a small tolerance. A maximum-iteration limit is included as a safety guard. If a cluster becomes empty, its previous centroid is retained.


In [ ]:
# Q26: Complete K-Means implementation from first principles.
def kmeans_from_scratch(p, data_points, initial_centroids, max_iter=300, tol=1e-8):
    data_points = np.asarray(data_points, dtype=float)
    centroids = np.asarray(initial_centroids, dtype=float).copy()

    if data_points.ndim != 2:
        raise ValueError("data_points must be a 2D array.")
    if centroids.shape != (p, data_points.shape[1]):
        raise ValueError("initial_centroids must have shape (p, number_of_features).")
    if p < 1 or p > len(data_points):
        raise ValueError("p must be between 1 and the number of data points.")

    for iteration in range(max_iter):
        # Assignment step: group each point with its nearest centroid.
        cluster_dict = {cluster_id: [] for cluster_id in range(p)}

        for point in data_points:
            assigned_cluster = closest_centroid_index(point, centroids)
            cluster_dict[assigned_cluster].append(point)

        # Update step: calculate means, preserving centroids of empty clusters.
        updated_centroids = calculate_new_centroids(cluster_dict)

        for cluster_id in range(p):
            if np.isnan(updated_centroids[cluster_id]).any():
                updated_centroids[cluster_id] = centroids[cluster_id]

        # Convergence check.
        centroid_shift = np.linalg.norm(updated_centroids - centroids)
        centroids = updated_centroids

        if centroid_shift <= tol:
            break

    # Reassign all points using the final centroids to return consistent clusters.
    final_clusters = {cluster_id: [] for cluster_id in range(p)}
    for point in data_points:
        assigned_cluster = closest_centroid_index(point, centroids)
        final_clusters[assigned_cluster].append(point)

    return centroids, final_clusters

# Example run with three clusters.
initial_centroids_3 = initialize_centroids(42, 3)
final_centroids_3, final_clusters_3 = kmeans_from_scratch(
    3, points, initial_centroids_3
)

print("Final centroids:")
print(final_centroids_3)
print("\nCluster sizes:")
for cluster_id, cluster_points in final_clusters_3.items():
    print(f"Cluster {cluster_id}: {len(cluster_points)} points")
print("SSE:", calculate_sse(final_centroids_3, final_clusters_3, 3))

### Q27 (6 marks): Run K-Means for k=1, 2, 3, 4; calculate and plot SSE

Each value of `k` uses initialization seed `42`. The elbow plot helps identify a value of `k` after which adding more clusters gives a smaller marginal reduction in SSE.


In [ ]:
# Q27: Compute SSE for k = 1, 2, 3, 4 using the scratch implementation.
k_values = [1, 2, 3, 4]
sse_values = []

for k in k_values:
    initial_centroids = initialize_centroids(42, k)
    final_centroids, final_clusters = kmeans_from_scratch(
        k, points, initial_centroids
    )
    sse = calculate_sse(final_centroids, final_clusters, k)
    sse_values.append(sse)
    print(f"k={k}, SSE={sse:.4f}")

print("\nList of SSE values:", sse_values)

plt.figure(figsize=(7, 5))
plt.plot(k_values, sse_values, marker="o", linewidth=2)
plt.xticks(k_values)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Sum of Squared Errors (SSE)")
plt.title("Elbow Method for K-Means")
plt.tight_layout()
plt.show()

### Q28 (6 marks): Apply K-Means using the elbow choice and visualize clusters and centroids

For this Iris two-feature dataset, `k=3` is a reasonable elbow choice and also aligns with the three known Iris species. The code below sets `best_k = 3` after inspecting the Q27 elbow plot. If your executed plot suggests a different elbow, change `best_k` accordingly.

The colors in this plot represent **unsupervised cluster assignments**, not the original Iris species labels.


In [ ]:
# Q28: Select the elbow value after inspecting the Q27 plot.
best_k = 3  # Change this if your elbow plot supports a different value.

best_initial_centroids = initialize_centroids(42, best_k)
best_centroids, best_clusters = kmeans_from_scratch(
    best_k, points, best_initial_centroids
)

# Convert cluster dictionary into arrays for plotting.
plt.figure(figsize=(8, 6))
for cluster_id in range(best_k):
    cluster_array = np.asarray(best_clusters[cluster_id], dtype=float)
    if len(cluster_array) > 0:
        plt.scatter(
            cluster_array[:, 0], cluster_array[:, 1],
            s=45, label=f"Cluster {cluster_id}"
        )

# Plot final centroids with a distinct marker.
plt.scatter(
    best_centroids[:, 0], best_centroids[:, 1],
    marker="X", s=250, c="black", edgecolors="white",
    linewidths=1.5, label="Centroids"
)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title(f"K-Means Clustering of Iris Petal Features (k={best_k})")
plt.legend()
plt.tight_layout()
plt.show()

print("Selected k:", best_k)
print("Final centroids:")
print(best_centroids)
print("Final SSE:", calculate_sse(best_centroids, best_clusters, best_k))

## Final checklist

- [x] Q1–Q5: Linear Regression
- [x] Q6–Q10: Logistic Regression
- [x] Q11–Q15: Support Vector Classifier
- [x] Q16–Q21: Bagging with bootstrap resampling and ensemble averaging
- [x] Q22–Q28: K-Means implemented from scratch

## Submission reminder

Run the notebook from top to bottom using **Restart and Run All**. Check that every output and plot is produced without errors, then download/save the completed notebook as an `.ipynb` file for submission.

**Academic integrity:** Review the code and explanations, make sure you understand the implementation, and follow your course's rules for permitted assistance.
